In [0]:
dbutils.widgets.text("catalog","fk_ce1366")
catalog = dbutils.widgets.get("catalog")

In [0]:

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.ops.time_travel_check AS
SELECT 
    COALESCE(a.product_id, b.product_id) AS product_id,
    CAST(a.selling_price AS DECIMAL(10,2)) AS selling_price_before_batch2,
    CAST(b.selling_price AS DECIMAL(10,2)) AS selling_price_current
FROM {catalog}.silver.products VERSION AS OF 3 a
FULL OUTER JOIN {catalog}.silver.products b
ON a.product_id = b.product_id
WHERE COALESCE(a.product_id, b.product_id) IN ('P004', 'P021', 'P050')
""")

spark.sql(f"SELECT * FROM {catalog}.ops.time_travel_check ORDER BY product_id").display()

In [0]:
#change data feed
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.ops.product_changes_batch2 AS
SELECT 
    a.product_id,
    a._change_type,
    a.mrp AS mrp_after_batch2,
    a.selling_price,
    a.is_active ,
    a._commit_version
FROM table_changes('{catalog}.silver.products', 2, 3) a
LEFT ANTI JOIN {catalog}.ops.time_travel_check b
ON a.product_id = b.product_id
WHERE a._commit_version = (SELECT MAX(_commit_version) FROM table_changes('{catalog}.silver.products', 2, 3))""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.ops.time_travel_check AS
SELECT 
    COALESCE(a.product_id, b.product_id) AS product_id,
    CAST(a.selling_price AS DECIMAL(10,2)) AS selling_price_before_batch2,
    CAST(b.selling_price AS DECIMAL(10,2)) AS selling_price_current
FROM {catalog}.silver.products VERSION AS OF 3 a
FULL OUTER JOIN {catalog}.silver.products b
ON a.product_id = b.product_id
""")
     

#change data feed
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.ops.product_changes_batch2 AS
SELECT 
    a.product_id,
    a._change_type,
    a.mrp AS mrp_after_batch2,
    a.selling_price,
    a.is_active ,
    a._commit_version
FROM table_changes('{catalog}.silver.products', 2, 3) a
LEFT ANTI JOIN {catalog}.ops.time_travel_check b
ON a.product_id = b.product_id
WHERE a._commit_version = (SELECT MAX(_commit_version) FROM table_changes('{catalog}.silver.products', 2, 3))""")